In [ ]:
#| default_exp renderers.comfyui

# renderers.comfyui

> Renderer for a local ComfyUI server via its REST API.
>
> Supports LoRA, negative prompts, and reference image input (via IP-Adapter or
> similar nodes in the workflow). Requires a running ComfyUI instance and a
> workflow JSON exported in API format.
>
> Config: `renderer.comfyui.base_url` and `renderer.comfyui.workflow_template_path`.

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations
import json
import time
import uuid
from pathlib import Path

import httpx

from manhualizer.config import OutputConfig, RendererConfig
from manhualizer.models import Panel, RenderResult
from manhualizer.render import BaseRenderer, ModelSpec

In [ ]:
#| export
# Node titles used to patch the TextToImage workflow template.
# These must match the _meta.title values in your ComfyUI workflow.
_PROMPT_NODE_TITLE = "positive_prompt"
_NEGATIVE_NODE_TITLE = "negative_prompt"
_WIDTH_NODE_TITLE = "empty_latent_image"       # direct width/height inputs
_WIDTH_PRIMITIVE_NODE_TITLE = "Width"           # PrimitiveInt node for width
_HEIGHT_PRIMITIVE_NODE_TITLE = "Height"         # PrimitiveInt node for height
_LORA_NODE_TITLE = "lora_loader"
_REF_IMAGE_NODE_TITLE = "reference_image"

# Node titles used to patch the SpeechBubble workflow template.
_SPEECH_BUBBLE_INPUT_IMAGE_TITLE = "bubble_input_image"
_SPEECH_BUBBLE_TEXT_TITLE = "bubble_text"

In [ ]:
#| export
class ComfyUIRenderer(BaseRenderer):
    """Image generation via a local ComfyUI server.

    The panel image is generated by the TextToImage workflow. Speech bubbles
    are applied as a separate post-processing step via ``add_speech_bubbles_batch_async``.

    Config:
        renderer.comfyui.base_url                    — ComfyUI server URL
        renderer.comfyui.workflow_template_path       — TextToImage workflow JSON
        renderer.comfyui.speech_bubble_workflow_path  — Speech bubble workflow JSON (optional)
        renderer.comfyui.poll_interval               — seconds between status checks
        renderer.comfyui.timeout                     — max wait seconds per panel
    """

    def __init__(self, model_spec: ModelSpec, config: RendererConfig):
        super().__init__(model_spec, config)
        self._model_cfg = config.comfyui
        self._workflow_template: dict | None = None
        self._speech_bubble_template: dict | None = None

    # ── Workflow loading ─────────────────────────────────────────────────────

    def _load_workflow(self) -> dict:
        if self._workflow_template is None:
            path = Path(self._model_cfg.workflow_template_path)
            if not path.exists():
                raise FileNotFoundError(
                    f"ComfyUI workflow not found: {path}. "
                    f"Set renderer.comfyui.workflow_template_path in your config."
                )
            self._workflow_template = json.loads(path.read_text())
        return json.loads(json.dumps(self._workflow_template))  # deep copy

    def _load_speech_bubble_workflow(self) -> dict | None:
        """Load the speech bubble workflow template, or None if not configured."""
        path_str = self._model_cfg.speech_bubble_workflow_path
        if not path_str:
            return None
        path = Path(path_str)
        if not path.exists():
            return None
        if self._speech_bubble_template is None:
            self._speech_bubble_template = json.loads(path.read_text())
        return json.loads(json.dumps(self._speech_bubble_template))  # deep copy

    # ── Workflow patching ────────────────────────────────────────────────────

    def _build_patch_map(
        self,
        panel: Panel,
        w: int,
        h: int,
        negative_prompt: str,
        reference_image_b64: str | None,
    ) -> dict:
        """Return a {node_title: patch_fn(inputs)} map for the TextToImage workflow."""
        patches: dict = {
            _PROMPT_NODE_TITLE:           lambda inp: inp.__setitem__("text", panel.visual_prompt) if "text" in inp else None,
            _NEGATIVE_NODE_TITLE:         lambda inp: inp.__setitem__("text", negative_prompt) if "text" in inp else None,
            _WIDTH_NODE_TITLE:            lambda inp: inp.update({k: v for k, v in {"width": w, "height": h}.items() if k in inp}),
            _WIDTH_PRIMITIVE_NODE_TITLE:  lambda inp: inp.__setitem__("value", w) if "value" in inp else None,
            _HEIGHT_PRIMITIVE_NODE_TITLE: lambda inp: inp.__setitem__("value", h) if "value" in inp else None,
        }
        if self.config.loras:
            lora = self.config.loras[0]
            patches[_LORA_NODE_TITLE] = lambda inp: inp.update(
                {k: v for k, v in {"lora_name": lora.path, "strength_model": lora.strength}.items() if k in inp}
            )
        if reference_image_b64:
            patches[_REF_IMAGE_NODE_TITLE] = lambda inp: inp.__setitem__("image", reference_image_b64) if "image" in inp else None
        return patches

    def _patch_workflow(
        self,
        workflow: dict,
        panel: Panel,
        output_cfg: OutputConfig,
        negative_prompt: str = "",
        reference_image_b64: str | None = None,
    ) -> dict:
        """Patch TextToImage workflow nodes with panel-specific values."""
        w, h = output_cfg.resolved_dimensions()
        patches = self._build_patch_map(panel, w, h, negative_prompt, reference_image_b64)
        for node in workflow.values():
            title = node.get("_meta", {}).get("title", "")
            patch_fn = patches.get(title)
            if patch_fn:
                patch_fn(node.get("inputs", {}))
        return workflow

    def _patch_speech_bubble_workflow(
        self,
        workflow: dict,
        image_filename: str,
        dialogue_text: str,
    ) -> dict:
        """Patch the speech bubble workflow with the panel image and dialogue."""
        for node in workflow.values():
            title = node.get("_meta", {}).get("title", "")
            inputs = node.get("inputs", {})
            if title == _SPEECH_BUBBLE_INPUT_IMAGE_TITLE and "image" in inputs:
                inputs["image"] = image_filename
            elif title == _SPEECH_BUBBLE_TEXT_TITLE and "text" in inputs:
                inputs["text"] = dialogue_text
        return workflow

    # ── ComfyUI HTTP helpers ─────────────────────────────────────────────────

    def _submit_and_wait(self, workflow: dict) -> bytes:
        """Submit a workflow to ComfyUI and poll until the image is ready."""
        base = self._model_cfg.base_url.rstrip("/")
        client_id = str(uuid.uuid4())

        resp = httpx.post(
            f"{base}/prompt",
            json={"prompt": workflow, "client_id": client_id},
            timeout=30,
        )
        resp.raise_for_status()
        prompt_id = resp.json()["prompt_id"]

        deadline = time.time() + self._model_cfg.timeout
        while time.time() < deadline:
            time.sleep(self._model_cfg.poll_interval)
            history = httpx.get(f"{base}/history/{prompt_id}", timeout=10).json()
            if prompt_id in history:
                outputs = history[prompt_id].get("outputs", {})
                for node_output in outputs.values():
                    images = node_output.get("images", [])
                    if images:
                        img_info = images[0]
                        img_resp = httpx.get(
                            f"{base}/view",
                            params={"filename": img_info["filename"],
                                    "subfolder": img_info.get("subfolder", ""),
                                    "type": img_info.get("type", "output")},
                            timeout=30,
                        )
                        img_resp.raise_for_status()
                        return img_resp.content

        raise TimeoutError(f"ComfyUI job {prompt_id} did not complete within {self._model_cfg.timeout}s")

    def _upload_image(self, image_path: Path) -> str:
        """Upload a local image to ComfyUI's input folder. Returns the filename."""
        base = self._model_cfg.base_url.rstrip("/")
        with open(image_path, "rb") as f:
            resp = httpx.post(
                f"{base}/upload/image",
                files={"image": (image_path.name, f, "image/png")},
                timeout=30,
            )
        resp.raise_for_status()
        return resp.json()["name"]

    # ── Speech bubble step ───────────────────────────────────────────────────

    def _apply_speech_bubbles(self, panel: Panel, image_path: Path) -> bytes | None:
        """Run the speech bubble workflow on the given panel image.

        Returns the new image bytes, or None if no workflow is configured.
        """
        workflow = self._load_speech_bubble_workflow()
        if workflow is None:
            return None

        image_filename = self._upload_image(image_path)
        dialogue_text = "\n".join(
            f"[{b.speaker}] ({b.bubble_type}): {b.text}"
            for b in panel.dialogue
        ) if panel.dialogue else ""

        workflow = self._patch_speech_bubble_workflow(workflow, image_filename, dialogue_text)
        return self._submit_and_wait(workflow)

    # ── Batch speech bubbles ─────────────────────────────────────────────────

    async def add_speech_bubbles_batch_async(
        self,
        panels: list,
        panels_dir: Path,
        resume: bool = True,
    ) -> list[Path]:
        """Apply speech bubbles to a batch of already-rendered panel images.

        Looks up each panel image in ``panels_dir`` by panel number, uploads it
        to ComfyUI, runs the speech bubble workflow on top, and overwrites the
        file in-place.  Panels with no dialogue are silently skipped.

        Args:
            panels: List of Panel objects (from storyboard).
            panels_dir: Directory containing the rendered panel images.
            resume: If True, skip panels whose images already have a
                    ``<stem>.bubbles_done`` marker file next to them.

        Returns:
            List of image Paths that were updated.

        Raises:
            FileNotFoundError: If no speech bubble workflow is configured.
        """
        import asyncio

        if self._load_speech_bubble_workflow() is None:
            raise FileNotFoundError(
                "No speech bubble workflow configured. "
                "Set renderer.comfyui.speech_bubble_workflow_path in your config."
            )

        sem = asyncio.Semaphore(2)  # ComfyUI handles one job at a time; keep low

        async def _process(panel) -> Path | None:
            if not panel.dialogue:
                return None

            # Find the panel image (try common formats)
            img_path: Path | None = None
            for ext in ("png", "jpg", "webp"):
                candidate = panels_dir / f"panel_{panel.panel_number:04d}.{ext}"
                if candidate.exists():
                    img_path = candidate
                    break
            if img_path is None:
                return None

            marker = img_path.with_suffix(".bubbles_done")
            if resume and marker.exists():
                return img_path

            async with sem:
                bubble_bytes = await asyncio.get_event_loop().run_in_executor(
                    None, self._apply_speech_bubbles, panel, img_path
                )
            if bubble_bytes is None:
                return None
            img_path.write_bytes(bubble_bytes)
            marker.touch()
            return img_path

        results = await asyncio.gather(*[_process(p) for p in panels])
        return [r for r in results if r is not None]

    # ── Rendering ────────────────────────────────────────────────────────────

    async def render_async(
        self,
        panel: Panel,
        output_dir: Path,
        output_cfg: OutputConfig,
        reference_images: dict[str, Path] | None = None,
        negative_prompt: str = "",
    ) -> RenderResult:
        import asyncio
        return await asyncio.get_event_loop().run_in_executor(
            None, self._render_sync, panel, output_dir, output_cfg, reference_images, negative_prompt
        )

    def _render_sync(
        self,
        panel: Panel,
        output_dir: Path,
        output_cfg: OutputConfig,
        reference_images: dict[str, Path] | None,
        negative_prompt: str = "",
    ) -> RenderResult:
        import base64

        workflow = self._load_workflow()

        ref_b64: str | None = None
        if reference_images:
            for char_name in panel.characters_present:
                ref_path = reference_images.get(char_name)
                if ref_path and ref_path.exists():
                    ref_b64 = base64.b64encode(ref_path.read_bytes()).decode()
                    break

        workflow = self._patch_workflow(workflow, panel, output_cfg, negative_prompt, ref_b64)
        image_bytes = self._submit_and_wait(workflow)

        out_path = output_dir / f"panel_{panel.panel_number:04d}.{output_cfg.format}"
        out_path.write_bytes(image_bytes)

        return RenderResult(
            panel_number=panel.panel_number,
            image_path=out_path,
            backend_used=self.model_spec.name,
            prompt_used=panel.visual_prompt,
            metadata={
                "ref_image_used": ref_b64 is not None,
                "loras": len(self.config.loras),
            },
        )

In [ ]:
# Construction and workflow patching tests (no server needed)
import json
from manhualizer.render import MODELS
from manhualizer.config import RendererConfig, LoRAConfig, OutputConfig
from manhualizer.models import Panel
from manhualizer.renderers.comfyui import ComfyUIRenderer

renderer = ComfyUIRenderer(MODELS["comfyui"], RendererConfig())
assert renderer.model_spec.capabilities.lora
assert renderer.model_spec.capabilities.reference_images
assert renderer.model_spec.capabilities.multi_image_input

# Patch a minimal mock workflow
mock_workflow = {
    "1": {"_meta": {"title": "positive_prompt"}, "inputs": {"text": "old prompt"}},
    "2": {"_meta": {"title": "negative_prompt"}, "inputs": {"text": ""}},
    "3": {"_meta": {"title": "empty_latent_image"}, "inputs": {"width": 512, "height": 512}},
}
renderer._workflow_template = mock_workflow

panel = Panel(panel_number=1, scene_id="s1", location="Forest",
              action_description="walks", visual_prompt="manhua style, forest scene")
out_cfg = OutputConfig(width=896, height=1152)

patched = renderer._patch_workflow(
    json.loads(json.dumps(mock_workflow)), panel, out_cfg, negative_prompt="blurry"
)
assert patched["1"]["inputs"]["text"] == "manhua style, forest scene"
assert patched["2"]["inputs"]["text"] == "blurry"
assert patched["3"]["inputs"]["width"] == 896
assert patched["3"]["inputs"]["height"] == 1152

print("ComfyUIRenderer OK")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()